# Download the `DT` rows of `validatielijst.csv`

Every row of `validatielijst.csv` whose `process` column is `DT` is downloaded from Waterinfo for its
locations, the period `startdatum` - `einddatum`, its `grootheid_code` and its `monstercompartiment_code`.
The result is saved as `<databundelcode>_<download date>.csv` in `data/waterinfo` of the repository,
in the same layout as `test_file.csv`.

Both CSV files are read from GitHub. **`GITHUB_BRANCH` is temporary: change it to `"main"` after the merge.**

In [14]:
from datetime import date
from io import StringIO
from pathlib import Path

import pandas as pd
import rws_waterinfo as rw
import urllib3

# TEMPORARY: development branch, change to "main" once merged (see downloading-dev/README.md)
GITHUB_BRANCH = "waterinfo_downloading"
GITHUB_BASE_URL = f"https://raw.githubusercontent.com/openearth/krmvalidatie/refs/heads/{GITHUB_BRANCH}/data"


def get_data_from_github(filename, encoding="windows-1252", delimiter=";"):
    # same approach as github_functions.py in the validation lambda
    url = f"{GITHUB_BASE_URL}/{filename}"
    response = urllib3.PoolManager().request("GET", url)
    if response.status != 200:
        raise RuntimeError(f"Failed to fetch {url}: HTTP {response.status} (is the file pushed to branch '{GITHUB_BRANCH}'?)")
    df = pd.read_csv(StringIO(response.data.decode(encoding)), delimiter=delimiter, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    return df

## 1. Read and check the CSV files

In [15]:
validatielijst = get_data_from_github("validatielijst.csv")
location_codes_df = get_data_from_github("waterinfo_location_codes.csv")

required = {
    "validatielijst.csv": (validatielijst, ["databundelcode", "grootheid_code", "monstercompartiment_code", "locatiecode", "startdatum", "einddatum", "process"]),
    "waterinfo_location_codes.csv": (location_codes_df, ["locatiecode", "waterinfo_code"]),
}
for name, (df, columns) in required.items():
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name} on branch '{GITHUB_BRANCH}' lacks column(s) {missing}. Found: {list(df.columns)}")

# names used in validatielijst -> location codes in the Waterinfo catalog
location_codes = dict(zip(location_codes_df["locatiecode"].str.strip(), location_codes_df["waterinfo_code"].str.strip()))
print(location_codes)

to_download = validatielijst[validatielijst["process"].fillna("").str.strip().str.upper() == "DT"]
print(f"{len(to_download)} row(s) with process = DT")
to_download[["databundelcode", "grootheid_code", "monstercompartiment_code", "locatiecode", "startdatum", "einddatum"]]

{'strand_Bergen': 'bergenaanzee.standafvalmeetnet', 'strand_Noordwijk': 'noordwijk.strandafvalmeetnet', 'strand_Terschelling': 'terschelling.strandafvalmeetnet', 'strand_Veere': 'veere.strandafvalmeetnet'}
1 row(s) with process = DT


,databundelcode,grootheid_code,monstercompartiment_code,locatiecode,startdatum,einddatum
559,RWS_2025_10 zwerfvuil op strand,AANTPLTE,OR,strand_Bergen;strand_Veere;strand_Noordwijk;st...,1-1-2025,31-12-2025


## 2. Download and save

In [16]:
result = None

# saved in <repository>/data/waterinfo, where the other downloaded Waterinfo data lives
out_dir = Path("../../../../data/waterinfo/notebook")
out_dir.mkdir(parents=True, exist_ok=True)

for _, row in to_download.iterrows():
    start = pd.to_datetime(row["startdatum"], dayfirst=True).strftime("%Y-%m-%d")
    end = pd.to_datetime(row["einddatum"], dayfirst=True).strftime("%Y-%m-%d")
    grootheid = row["grootheid_code"].strip()
    compartiment = row["monstercompartiment_code"].strip()

    names = [n.strip() for n in row["locatiecode"].split(";")]
    unknown = [n for n in names if n not in location_codes]
    if unknown:
        raise ValueError(f"No Waterinfo code known for {unknown}, add them to waterinfo_location_codes.csv")

    frames = []
    for name in names:
        loc = location_codes[name]
        params = [[compartiment, "", "", grootheid, loc, start, end]]
        df = rw.get_data(params=params, return_df=True, max_workers=8)
        print(f"{loc}: {len(df)} rows")
        frames.append(df)

    result = pd.concat(frames, ignore_index=True)
    filename = f"{row['databundelcode'].strip()}_{date.today():%Y%m%d}.csv"
    result.to_csv(out_dir / filename, index=False, sep=";")
    print(f"{filename}: {len(result)} rows, {start} - {end}, {grootheid}, {compartiment}")
    print(f"saved in {out_dir.resolve()}")

result.head() if result is not None else print("Nothing to download")

Downloaded 1/1
bergenaanzee.standafvalmeetnet: 584 rows
Downloaded 1/1
veere.strandafvalmeetnet: 584 rows
Downloaded 1/1
noordwijk.strandafvalmeetnet: 584 rows
Downloaded 1/1
terschelling.strandafvalmeetnet: 584 rows
RWS_2025_10 zwerfvuil op strand_20261015.csv: 2336 rows, 2025-01-01 - 2025-12-31, AANTPLTE, OR
saved in /work/data/waterinfo/notebook


,Code,Coordinatenstelsel,Lat,Lon,Naam,Omschrijving,Tijdstip,Meetwaarde.Waarde_Alfanumeriek,Meetwaarde.Waarde_Numeriek,WaarnemingMetadata.Bemonsteringshoogte,...,Parameter.Omschrijving,Typering.Code,Typering.Omschrijving,WaardeBepalingsMethode.Code,WaardeBepalingsMethode.Omschrijving,WaardeBepalingsTechniek.Code,WaardeBepalingsTechniek.Omschrijving,WaardeBewerkingsMethode.Code,WaardeBewerkingsMethode.Omschrijving,Meetwaarde.Waarde_Limietsymbool
0,bergenaanzee.standafvalmeetnet,ETRS89,52.651355,4.626157,"Bergen aan Zee, Standafval Meetnet",NL001,2025-01-14T12:00:00.000+01:00,0,0.0,0,...,Ander textiel voorwerp,NaN,NaN,other:X156,Guideline for monitoring marine litter on beac...,NaN,NaN,NaN,NaN,<NA>
1,bergenaanzee.standafvalmeetnet,ETRS89,52.651355,4.626157,"Bergen aan Zee, Standafval Meetnet",NL001,2025-04-16T12:00:00.000+01:00,0,0.0,0,...,Ander textiel voorwerp,NaN,NaN,other:X156,Guideline for monitoring marine litter on beac...,NaN,NaN,NaN,NaN,<NA>
2,bergenaanzee.standafvalmeetnet,ETRS89,52.651355,4.626157,"Bergen aan Zee, Standafval Meetnet",NL001,2025-07-10T12:00:00.000+01:00,1,1.0,0,...,Ander textiel voorwerp,NaN,NaN,other:X156,Guideline for monitoring marine litter on beac...,NaN,NaN,NaN,NaN,<NA>
3,bergenaanzee.standafvalmeetnet,ETRS89,52.651355,4.626157,"Bergen aan Zee, Standafval Meetnet",NL001,2025-10-31T12:00:00.000+01:00,0,0.0,0,...,Ander textiel voorwerp,NaN,NaN,other:X156,Guideline for monitoring marine litter on beac...,NaN,NaN,NaN,NaN,<NA>
4,bergenaanzee.standafvalmeetnet,ETRS89,52.651355,4.626157,"Bergen aan Zee, Standafval Meetnet",NL001,2025-01-14T12:00:00.000+01:00,0,0.0,0,...,Papieren zak,NaN,NaN,other:X156,Guideline for monitoring marine litter on beac...,NaN,NaN,NaN,NaN,<NA>
